## Load latest headlines

In [1]:
# ========== Bootstrap: ensure required Python packages are present =========
import importlib, subprocess, sys
from typing import Optional

def _ensure(pkg_name: str, import_name: Optional[str] = None):
    try:
        importlib.import_module(import_name or pkg_name)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg_name])
    finally:
        globals()[import_name or pkg_name] = importlib.import_module(import_name or pkg_name)

_ensure('pandas')
print('All dependencies ready.\n')


All dependencies ready.



In [2]:
from pathlib import Path
archive = Path('archive')
archive.mkdir(exist_ok=True)
import pandas as pd
latest = pd.read_csv('../headlines/latest.csv')
latest.head()

,pubdate,source,title,link
0,2026-11-05-20-55-11 +0000,nypost,San Francisco restaurant closes after 30-year ...,https://nypost.com/2026/11/05/business/san-fra...
1,2026-11-05-20-54-12 +0000,nypost,Nithya Raman pays tribute to Karen Bass and th...,https://nypost.com/2026/11/05/us-news/nithya-r...
2,2026-11-05-20-53-58 +0000,nypost,LAPD fears crime ‘nightmare’ as Nithya Raman e...,https://nypost.com/2026/11/05/us-news/la-polic...
3,2026-11-05-20-53-33 +0000,nypost,Key group that helped Trump win in 2024 drove ...,https://nypost.com/2026/11/05/us-news/latino-m...
4,2026-11-05-20-52-46 +0000,nypost,"Dad, 34, made desperate call to wife moments ...",https://nypost.com/2026/11/05/us-news/dad-34-m...


## Calculate word frequencies

In [3]:
import re
from collections import Counter
from datetime import datetime

with open('exclude.txt') as f:
    stop_words = set(w.strip() for w in f if w.strip())
words = re.findall(r'[A-Za-z]+', ' '.join(latest['title']).lower())
filtered = [w for w in words if w not in stop_words and len(w) > 1]
counts = Counter(filtered)
score_df = (
    pd.DataFrame(counts.items(), columns=['word','score'])
    .sort_values('score', ascending=False)
)
score_df[['score','word']].to_csv('scores.csv', index=False)
timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')
score_df[['score','word']].to_csv(archive / f'scores-{timestamp}.csv', index=False)
score_df.head()


/tmp/ipykernel_2681/2220774760.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')


,word,score
32,trump,43
368,democrats,25
334,midterm,20
198,new,19
83,will,16


## Rank headlines by score

In [4]:
word_scores = dict(score_df[['word','score']].values)
latest['score'] = latest['title'].apply(
    lambda t: sum(
        word_scores.get(w.lower(), 0)
        for w in re.findall(r'[A-Za-z]+', t)
        if len(w) > 1
    )
)
ranked = latest.sort_values('score', ascending=False)
ranked[['score','pubdate','source','title','link']].to_csv('rank.csv', index=False)
ranked[['score','pubdate','source','title','link']].to_csv(archive / f'rank-{timestamp}.csv', index=False)
ranked.head()


,pubdate,source,title,link,score
412,2026-11-04-21-21-00 +0000,wsj,Markets surged Wednesday after American voters...,https://www.wsj.com/politics/elections/the-win...,138
129,2026-11-05-17-20-00 +0000,wsj,President Trump put three justices on the Supr...,https://www.wsj.com/politics/policy/midterm-ro...,134
383,2026-11-04-22-37-00 +0000,wsj,Democrats’ midterm sweep has shaken up the 202...,https://www.wsj.com/politics/elections/democra...,133
126,2026-11-05-17-23-49 +0000,bbc,Farage says he is worried Trump will step up I...,https://www.bbc.co.uk/news/articles/cqz0vz55jp...,129
259,2026-11-05-11-00-00 +0000,latimes,"Hiltzik: With Trump and GOP on the run, Democr...",https://www.latimes.com/business/story/2026-11...,128


## Select top headlines

In [5]:
with open('exclude_sources.txt') as f:
    excluded_sources = {
        line.strip()
        for line in f
        if line.strip() and not line.strip().startswith('#')
    }

top_count = 10
top_rows = []
working = word_scores.copy()
remaining = latest.copy()

while len(top_rows) < top_count and not remaining.empty:
    scored_remaining = remaining.assign(
        score=remaining['title'].apply(
            lambda t: sum(
                working.get(w.lower(), 0)
                for w in re.findall(r'[A-Za-z]+', t)
                if len(w) > 1
            )
        )
    )
    ranked_loop = scored_remaining.sort_values('score', ascending=False)

    if ranked_loop.empty:
        break

    top_story = ranked_loop.iloc[0]
    remaining = remaining.drop(top_story.name)

    if top_story['source'] in excluded_sources:
        continue

    words = set(re.findall(r'[A-Za-z]+', top_story['title'].lower()))
    for word in words:
        working.pop(word, None)

    top_rows.append(top_story[['score', 'pubdate', 'source', 'title', 'link']])

top_df = pd.DataFrame(top_rows)
top_df.to_csv('top.csv', index=False)
top_df.to_csv(archive / f'top-{timestamp}.csv', index=False)
top_df


,score,pubdate,source,title,link
412,138,2026-11-04-21-21-00 +0000,wsj,Markets surged Wednesday after American voters...,https://www.wsj.com/politics/elections/the-win...
383,131,2026-11-04-22-37-00 +0000,wsj,Democrats’ midterm sweep has shaken up the 202...,https://www.wsj.com/politics/elections/democra...
207,77,2026-11-05-13-26-00 +0000,wsj,The Bank of England raised its key interest ra...,https://www.wsj.com/economy/central-banking/ba...
53,74,2026-11-05-19-41-43 +0000,nypost,Karen Bass calls Nithya Raman to concede LA ma...,https://nypost.com/2026/11/05/us-news/karen-ba...
190,72,2026-11-05-14-23-16 +0000,nypost,Russell Brand convicted of rape in UK court — ...,https://nypost.com/2026/11/05/world-news/russe...
200,71,2026-11-05-13-54-14 +0000,nypost,"Democrat, 19, wins Massachusetts House seat by...",https://nypost.com/2026/11/05/us-news/democrat...
260,58,2026-11-05-11-00-00 +0000,latimes,New Culver City development will add hundreds ...,https://www.latimes.com/business/story/2026-11...
23,53,2026-11-05-20-28-38 +0000,bbc,California governor commutes death sentences f...,https://www.bbc.co.uk/news/articles/cqrkyyd2j8...
189,53,2026-11-05-14-25-00 +0000,wsj,The Democrats strolled to victory in the midte...,https://www.wsj.com/politics/policy/the-democr...
159,44,2026-11-05-15-53-00 +0000,wsj,"Bomb Threats Cause Diverted Flights, Emergency...",https://www.wsj.com/business/airlines/flights-...


## Create JSON version of top headlines

In [6]:
import pandas as pd
pd.read_csv('top.csv').to_json('top.json', orient='records', indent=2)
